In [ ]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

In [ ]:
%%cuda
#include <iostream>

__global__ void helloFromGPU() {
    printf("Hello World from the GPU!\n");
}

int main() {
    helloFromGPU<<<2, 5>>>();
    // Wait for the GPU to finish before accessing on host
    cudaDeviceSynchronize();
    return 0;
}


<h1> __global__ </h1>

The __global__ execution space specifier declares a function as being a kernel. Such a function is:

- Executed on the device,

- Callable from the host,

- Callable from the device for devices of compute capability 5.0 or higher (see CUDA Dynamic Parallelism for more details).

A __global__ function must have void return type, and cannot be a member of a class.

Any call to a __global__ function must specify its execution configuration as described in Execution Configuration.

A call to a __global__ function is asynchronous, meaning it returns before the device has completed its execution.



In [ ]:
%%cuda

#include <iostream>



__global__ void test01() {
    printf("calling this function from block %d and thread %d\n", blockIdx.x, threadIdx.x);
}

int main()
{
    test01<<<2, 3>>>();
    cudaDeviceSynchronize();
    return 0;
}

In [ ]:
%%cuda

#include <iostream>

__global__ void test02() {
    printf("calling this function from block %d and thread %d\n", blockIdx.x, threadIdx.x);
}

int main()
{
    test02<<<2, 2048>>>();
    cudaDeviceSynchronize();
    return 0;
}

Vector Addition

__global__ void vectorAdd(const int* A, const int* B, int* C, int n)
{
    int i = threadIdx.x;  // Get thread ID (runs on device)
    if(i < n)              // Boundary check (runs on device)
    {
        C[i] = A[i] + B[i];  // Parallel addition (runs on device)
    }
The `main` function runs on the host (CPU) and performs the following steps:

1. **Initialize arrays on host:**  
    `A`, `B`, and `C` are created and initialized in host memory.

2. **Allocate memory on device:**  
    `cudaMalloc` is used to allocate space for `d_A`, `d_B`, and `d_C` on the device (GPU).

3. **Copy data from host to device:**  
    `cudaMemcpy` transfers the contents of `A` and `B` from host memory to device memory.

4. **Launch kernel on device:**  
    `vectorAdd<<<1, n>>>(d_A, d_B, d_C, n);` launches the kernel with 1 block of `n` threads. Each thread computes one element of the sum in parallel on the device.

5. **Copy result from device to host:**  
    After computation, `cudaMemcpy` copies the result array `C` from device memory back to host memory.

6. **Print results on host:**  
    The host prints the contents of `C` to display the result of the vector addition.

7. **Free device memory:**  
    `cudaFree` releases the memory allocated on the device.

8. **Return from main:**  
    The program ends and returns control to the operating system.

All memory management and kernel launch commands are executed on the host, while the actual addition is performed in parallel on the device by the kernel.


In [ ]:
%%cuda

#include <iostream>
#include <cuda_runtime.h>

__global__ void vectorAdd(const int* A, const int* B, int* C, int n)
{
    int i = threadIdx.x;
    if(i < n)
    {
        C[i] = A[i] + B[i];
    }
}

int main()
{
    const int n = 5;
    int A[n] = {1,2,3,4,5};
    int B[n] = {10,20,30,40,50};
    int C[n] = {0};

    int *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, n * CHUNKSof(int));
    cudaMalloc(&d_B, n * sizeof(int));  
    cudaMalloc(&d_C, n * sizeof(int));

    cudaMemcpy(d_A, A, n * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, n * sizeof(int), cudaMemcpyHostToDevice);

    vectorAdd<<<1, n>>>(d_A, d_B, d_C, n);

    cudaMemcpy(C, d_C, n * sizeof(int), cudaMemcpyDeviceToHost);

    for(int i = 0; i < n; i++)
    {
        std::cout << C[i] << " ";
    }
    std::cout << std::endl;

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

In [ ]:
%%cuda

#include <iostream>
#include <cuda_runtime.h>

#define N 4100
#define THREADS_PER_BLOCK 1024

__global__ void vectorAdd(int* A, int* B, int* C, int* countBlocks, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if(i < n)
    {
        C[i] = A[i] + B[i];
        printf("Block %d processed element %d\n", blockIdx.x, i);
        countBlocks[blockIdx.x]++;
    }
}

int main()
{
    int *A, *B, *C;
    A = (int*)malloc(N*sizeof(int));
    B = (int*)malloc(N*sizeof(int));
    C = (int*)malloc(N*sizeof(int));

    for(int i = 0; i < N; i++)
    {
        A[i] = i;
        B[i] = N - i;
    }

    int *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, N*sizeof(int));
    cudaMalloc(&d_B, N*sizeof(int));
    cudaMalloc(&d_C, N*sizeof(int));

    cudaMemcpy(d_A, A, N*sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, N*sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_C, C, N*sizeof(int), cudaMemcpyHostToDevice);

    int blocksize = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    std::cout<<"blocksize: " << blocksize<<std::endl;

    int *countBlocks;
    cudaMalloc(&countBlocks, blocksize*sizeof(int));
    cudaMemset(countBlocks, 0, blocksize*sizeof(int));

    vectorAdd<<<blocksize, THREADS_PER_BLOCK>>>(d_A, d_B, d_C, countBlocks, N);
    cudaDeviceSynchronize();

    cudaMemcpy(C, d_C, N*sizeof(int), cudaMemcpyDeviceToHost);
    /*for(int i = 0; i < N; i++)
    {
        std::cout << C[i] << " ";
    }
    std::cout << std::endl;*/

    int *h_countBlocks = (int*)malloc(blocksize * sizeof(int));
    cudaMemcpy(h_countBlocks, countBlocks, blocksize * sizeof(int), cudaMemcpyDeviceToHost);

    for(int i = 0; i < blocksize; i++)
    {
        std::cout << "Block " << i << " processed " << h_countBlocks[i] << " elements." << std::endl;
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    cudaFree(countBlocks);
    free(A);
    free(B);
    free(C);

}



# Vector with large set of data

In [ ]:
%%cuda
#include <iostream>

#define SIZE 1024*1024*1024
#define CHUNKS 1024*1024*125
#define THREADS_PER_BLOCK 1024

__global__ void vectorAdd(int* A, int* B, int* C, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if(i < n)
    {
        C[i] = A[i] + B[i];
    }
}

int main()
{
    int *A, *B, *C;
    A = (int*)malloc(CHUNKS*sizeof(int));
    B = (int*)malloc(CHUNKS*sizeof(int));
    C = (int*)malloc(CHUNKS*sizeof(int));


    int *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, CHUNKS*sizeof(int));
    cudaMalloc(&d_B, CHUNKS*sizeof(int));
    cudaMalloc(&d_C, CHUNKS*sizeof(int));

    for(int i = 0; i < SIZE; i+= CHUNKS)
    {
        for(int j = 0; j < CHUNKS; j++)
        {
            A[j] = i + j;
            B[j] = CHUNKS - j;
        }
        cudaMemcpy(d_A, A, CHUNKS*sizeof(int), cudaMemcpyHostToDevice);
        cudaMemcpy(d_B, B, CHUNKS*sizeof(int), cudaMemcpyHostToDevice);
        cudaMemcpy(d_C, C, CHUNKS*sizeof(int), cudaMemcpyHostToDevice);

        int blocksize = (CHUNKS + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;

        vectorAdd<<<blocksize, THREADS_PER_BLOCK>>>(d_A, d_B, d_C, CHUNKS);
            cudaDeviceSynchronize();
        cudaMemcpy(C, d_C, CHUNKS*sizeof(int), cudaMemcpyDeviceToHost);
        for(int i = 0; i < 10; i++)
        {
            std::cout << C[i] << " ";
        }
        std::cout << std::endl;
    }



    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(A);
    free(B);
    free(C);
}